# Carregamento da Base Tratada

Nesta etapa é carregado o dataset após o processo de limpeza e classificação realizado na etapa de preparação dos dados.

A base contém documentos técnicos extraídos de repositórios, classificados por categoria e preparados para geração dos embeddings utilizados pelo modelo de IA.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

caminho_dataset = "/content/drive/MyDrive/Hackathon_ONE/dataset/tratado/documentation_tratado.parquet"

dataset = pd.read_parquet(caminho_dataset)

print("Dataset carregado!")
print(dataset.shape)

dataset.head()

In [ ]:
import pandas as pd

caminho = "/content/drive/MyDrive/Hackathon_ONE/dataset/tratado/documentation_tratado.parquet"

df = pd.read_parquet(caminho)

print(df.shape)
df.head()

In [ ]:
import sentence_transformers

print("Biblioteca encontrada!")

In [ ]:
df_teste = df.head(100).copy()

print(df_teste.shape)

In [ ]:
from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer("all-MiniLM-L6-v2")

print("Modelo carregado com sucesso!")

In [ ]:
df_readme = df[df["categoria_documento"] == "Documentacao"].copy()

print(df_readme.shape)

In [ ]:
# Seleciona apenas 10 documentos para teste
df_teste = df_readme.head(10).copy()

# Gera os embeddings
embeddings = modelo.encode(
    df_teste["content"].tolist(),
    show_progress_bar=True
)

print("Embeddings gerados com sucesso!")
print("Quantidade de embeddings:", len(embeddings))
print("Dimensão de cada embedding:", len(embeddings[0]))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

print("Biblioteca carregada!")

In [ ]:
similaridade = cosine_similarity(embeddings)

print(similaridade.shape)

similaridade

In [ ]:
import numpy as np

def recomendar_documentos(indice, top_n=3):
    similaridades = similaridade[indice]

    # Ordena do maior para o menor
    indices = np.argsort(similaridades)[::-1]

    # Remove o próprio documento
    indices = indices[indices != indice]

    print(f"\nDocumento consultado:\n")
    print(df_teste.iloc[indice]["repo_full_name"])
    print(df_teste.iloc[indice]["file_name"])

    print("\nDocumentos mais semelhantes:\n")

    for i in indices[:top_n]:
        print(f"Repositório: {df_teste.iloc[i]['repo_full_name']}")
        print(f"Arquivo: {df_teste.iloc[i]['file_name']}")
        print(f"Similaridade: {similaridades[i]:.3f}")
        print("-" * 50)

In [ ]:
recomendar_documentos(0)

# Geração dos Embeddings - SCRIPTO

In [ ]:
print(df.shape)
print(df_readme.shape)
print(type(modelo))

In [ ]:
# Gerando embeddings de todos os documentos técnicos

conteudos = df_readme["content"].tolist()

embeddings_readme = modelo.encode(
    conteudos,
    show_progress_bar=True,
    batch_size=32
)

print("Embeddings gerados com sucesso!")
print("Quantidade de documentos:", len(embeddings_readme))
print("Dimensão dos vetores:", len(embeddings_readme[0]))

In [ ]:
import os

caminho_base = "/content/drive/MyDrive/Hackathon_ONE/SCRIPTO_AI"

os.makedirs(f"{caminho_base}/embeddings", exist_ok=True)
os.makedirs(f"{caminho_base}/metadados", exist_ok=True)

print("Estrutura SCRIPTO_AI criada com sucesso!")

In [ ]:
import numpy as np

caminho_embeddings = "/content/drive/MyDrive/Hackathon_ONE/SCRIPTO_AI/embeddings/embeddings_readme.npy"

np.save(caminho_embeddings, embeddings_readme)

print("Embeddings salvos com sucesso!")

In [ ]:
metadados = df_readme[
    [
        "repo_full_name",
        "file_name",
        "categoria_documento",
        "content"
    ]
].copy()

metadados["indice_embedding"] = range(len(metadados))

caminho_metadados = "/content/drive/MyDrive/Hackathon_ONE/SCRIPTO_AI/metadados/documentos_readme.csv"

metadados.to_csv(caminho_metadados, index=False)

print("Metadados salvos com sucesso!")
print(metadados.shape)

In [ ]:
import os

caminhos = [
    "/content/drive/MyDrive/Hackathon_ONE/SCRIPTO_AI/embeddings/embeddings_readme.npy",
    "/content/drive/MyDrive/Hackathon_ONE/SCRIPTO_AI/metadados/documentos_readme.csv"
]

for arquivo in caminhos:
    print(arquivo)
    print("Existe:", os.path.exists(arquivo))
    print("-" * 40)

# Motor de Busca Semântica - SCRIPTO

Nesta etapa será criado o motor de busca inteligente do SCRIPTO.

Após a geração dos embeddings dos documentos técnicos, será realizada a preparação dos arquivos necessários para consulta.

O objetivo desta etapa é permitir que uma busca em linguagem natural seja comparada semanticamente com os documentos existentes, identificando conteúdos relacionados mesmo quando não possuem exatamente as mesmas palavras-chave.

Fluxo da etapa:

1. Carregar os embeddings gerados pelo modelo de IA;
2. Carregar os metadados dos documentos;
3. Associar cada vetor ao seu respectivo documento;
4. Preparar a base para cálculo de similaridade.

Os embeddings representam o conteúdo textual dos documentos em formato numérico, permitindo que o sistema encontre relações de significado entre diferentes conteúdos técnicos.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

# Carregar embeddings salvos
caminho_embeddings = "/content/drive/MyDrive/Hackathon_ONE/SCRIPTO_AI/embeddings/embeddings_readme.npy"

embeddings_base = np.load(caminho_embeddings)

# Carregar metadados
caminho_metadados = "/content/drive/MyDrive/Hackathon_ONE/SCRIPTO_AI/metadados/documentos_readme.csv"

metadados = pd.read_csv(caminho_metadados)

print("Base de IA carregada!")
print("Embeddings:", embeddings_base.shape)
print("Metadados:", metadados.shape)

# Função de Busca Semântica - SCRIPTO

Nesta etapa será desenvolvida a função responsável pela consulta inteligente dos documentos técnicos.

O usuário poderá realizar uma pesquisa utilizando linguagem natural e o modelo de IA irá comparar o significado da busca com os embeddings dos documentos existentes.

Funcionamento:

1. O texto informado pelo usuário será transformado em um embedding utilizando o mesmo modelo aplicado nos documentos;
2. Será calculada a similaridade entre a busca e todos os documentos armazenados;
3. Os documentos com maior proximidade semântica serão retornados como recomendação.

Essa abordagem permite encontrar conteúdos relacionados mesmo quando as palavras utilizadas na busca são diferentes das palavras presentes nos documentos.

Exemplo:

Busca:
"Como trabalhar com containers"

Pode encontrar documentos relacionados a:
"Docker", "containerização", "deploy de aplicações", entre outros.

In [ ]:
def buscar_scripto(pergunta, top_n=5):

    # Transformar a pergunta em embedding
    embedding_pergunta = modelo.encode([pergunta])

    # Calcular similaridade com todos os documentos
    similaridades = cosine_similarity(
        embedding_pergunta,
        embeddings_base
    )[0]

    # Pegar os maiores valores
    indices = np.argsort(similaridades)[::-1][:top_n]

    resultados = metadados.iloc[indices].copy()

    resultados["similaridade"] = similaridades[indices]

    return resultados[
        [
            "repo_full_name",
            "file_name",
            "categoria_documento",
            "similaridade"
        ]
    ]

# Teste do Motor de Busca Semântica - SCRIPTO

Nesta etapa será realizada a validação do modelo de IA através de consultas em linguagem natural.

O objetivo é verificar se o sistema consegue recuperar documentos tecnicamente relacionados ao termo pesquisado utilizando similaridade semântica.

Diferente de uma busca tradicional por palavras-chave, o modelo considera o contexto e o significado dos textos.

Exemplo:

Consulta:
"docker containers"

O sistema deve retornar documentos relacionados a containers, infraestrutura, configuração ou tecnologias próximas.

In [ ]:
buscar_scripto("docker containers")

# Padronização da Resposta para Integração com API

Nesta etapa será ajustado o formato de retorno da busca semântica.

O objetivo é disponibilizar uma estrutura mais simples para consumo pelo Backend do SCRIPTO.

A resposta será organizada contendo:

- Nome do projeto;
- Tipo do documento encontrado;
- Categoria do conteúdo;
- Score de similaridade.

Esse formato facilita a comunicação entre o modelo de IA desenvolvido em Python e a API desenvolvida em Spring Boot.

In [ ]:
def buscar_scripto(pergunta, top_n=5):

    embedding_pergunta = modelo.encode([pergunta])

    similaridades = cosine_similarity(
        embedding_pergunta,
        embeddings_base
    )[0]

    indices = np.argsort(similaridades)[::-1][:top_n]

    resultados = []

    for indice in indices:
        resultados.append({
            "projeto": metadados.iloc[indice]["repo_full_name"],
            "documento": metadados.iloc[indice]["file_name"],
            "categoria": metadados.iloc[indice]["categoria_documento"],
            "similaridade": round(float(similaridades[indice]) * 100, 2)
        })

    return resultados

# Conclusão - Modelo IA SCRIPTO

Nesta etapa foi desenvolvido o mecanismo de busca semântica do SCRIPTO.

O modelo de IA foi capaz de transformar documentos técnicos em representações vetoriais e realizar consultas baseadas em significado.

Principais entregas:

- 941 documentos processados;
- Embeddings gerados utilizando Sentence Transformers;
- Busca semântica implementada;
- Ranking por similaridade;
- Estrutura de resposta preparada para integração com API.

Os arquivos gerados podem ser utilizados pelo Backend para disponibilizar a funcionalidade de pesquisa inteligente no sistema SCRIPTO.

# Sugestão Automática de Conteúdos Relacionados - SCRIPTO

Nesta etapa será implementada uma funcionalidade adicional utilizando os embeddings já gerados.

O objetivo é recomendar documentos técnicos semelhantes ao conteúdo atualmente consultado pelo usuário.

Diferente da busca tradicional, essa recomendação utiliza similaridade semântica, analisando o contexto dos documentos para identificar relações entre conteúdos.

Fluxo:

1. Usuário seleciona um documento;
2. O sistema utiliza o embedding correspondente;
3. Calcula a similaridade com os demais documentos da base;
4. Remove o próprio documento da recomendação;
5. Retorna os conteúdos mais relacionados.

Essa funcionalidade amplia a capacidade do SCRIPTO, permitindo descoberta de conhecimento e reutilização de informações técnicas.

In [ ]:
def recomendar_documentos(indice_documento, top_n=5):

    # Embedding do documento selecionado
    embedding_documento = embeddings_base[indice_documento].reshape(1, -1)

    # Similaridade com todos os documentos
    similaridades = cosine_similarity(
        embedding_documento,
        embeddings_base
    )[0]

    # Ordenar do maior para o menor
    indices = np.argsort(similaridades)[::-1]

    resultados = []

    for indice in indices:

        # Não recomendar o próprio documento
        if indice == indice_documento:
            continue

        resultados.append({
            "projeto": metadados.iloc[indice]["repo_full_name"],
            "documento": metadados.iloc[indice]["file_name"],
            "categoria": metadados.iloc[indice]["categoria_documento"],
            "similaridade": round(float(similaridades[indice]) * 100, 2)
        })

        if len(resultados) == top_n:
            break

    return resultados

# Validação do Documento Selecionado

Antes de gerar recomendações, será validado qual documento está associado ao índice utilizado.

Essa verificação garante que o embedding selecionado corresponde corretamente ao documento apresentado ao usuário.

In [ ]:
metadados.iloc[0]

In [ ]:
metadados.sort_values(
    "content",
    key=lambda x: x.str.len(),
    ascending=False
).head(1)[
    [
        "repo_full_name",
        "file_name",
        "categoria_documento",
        "indice_embedding"
    ]
]

# Teste de Recomendação com Documento Técnico Extenso

Nesta etapa será validada a funcionalidade de recomendação utilizando um documento com maior volume de informações técnicas.

O objetivo é verificar se o modelo consegue identificar conteúdos relacionados a partir do contexto semântico de um documento completo.

Documento utilizado no teste:

- Repositório: sudheerj/reactjs-interview-questions
- Arquivo: README.md
- Índice do embedding: 836

A expectativa é que o modelo encontre documentos com temas próximos, como desenvolvimento web, programação, frameworks e tecnologias relacionadas.

In [ ]:
recomendar_documentos(836)

# Filtro de Relevância das Recomendações

Nesta etapa será aplicado um filtro mínimo de similaridade para melhorar a qualidade das recomendações.

O objetivo é evitar que documentos pouco relacionados sejam apresentados ao usuário.

Critério utilizado:

- Documentos com similaridade abaixo de 35% serão descartados;
- Apenas conteúdos considerados semanticamente relacionados serão retornados.

Essa etapa melhora a experiência do usuário e aumenta a precisão da recomendação.

In [ ]:
def recomendar_documentos(indice_documento, top_n=5, limite_similaridade=35):

    # Embedding do documento selecionado
    embedding_documento = embeddings_base[indice_documento].reshape(1, -1)

    # Calcula similaridade com todos os documentos
    similaridades = cosine_similarity(
        embedding_documento,
        embeddings_base
    )[0]

    # Ordena do maior para o menor
    indices = np.argsort(similaridades)[::-1]

    resultados = []

    for indice in indices:

        # Ignorar o próprio documento
        if indice == indice_documento:
            continue

        # Converter similaridade para percentual
        score = float(similaridades[indice]) * 100

        # Aplicar filtro de relevância
        if score < limite_similaridade:
            continue

        resultados.append({
            "projeto": metadados.iloc[indice]["repo_full_name"],
            "documento": metadados.iloc[indice]["file_name"],
            "categoria": metadados.iloc[indice]["categoria_documento"],
            "similaridade": round(score, 2)
        })

        if len(resultados) == top_n:
            break

    return resultados

# Apresentação dos Resultados - SCRIPTO

Nesta etapa será criada uma camada de apresentação dos resultados gerados pelo modelo de IA.

O objetivo é transformar a saída técnica da busca semântica em uma visualização mais amigável para usuários e para futura integração com a interface do sistema.

A lógica de recomendação permanece separada da camada de apresentação, permitindo que o Backend ou Frontend utilize os dados conforme a necessidade.

In [ ]:
def exibir_recomendacoes(resultados):

    print("🔎 Conteúdos relacionados encontrados\n")

    print("=" * 50)

    for posicao, item in enumerate(resultados, start=1):

        if posicao == 1:
            medalha = "🥇"
        elif posicao == 2:
            medalha = "🥈"
        elif posicao == 3:
            medalha = "🥉"
        else:
            medalha = "📌"

        print(f"{medalha} {posicao}º Resultado")
        print(f"📂 Projeto: {item['projeto']}")
        print(f"📄 Documento: {item['documento']}")
        print(f"🏷 Categoria: {item['categoria']}")
        print(f"⭐ Similaridade: {item['similaridade']}%")
        print("-" * 50)

In [ ]:
resultados = recomendar_documentos(836)

exibir_recomendacoes(resultados)

# Classificação de Relevância dos Resultados

Nesta etapa será criada uma classificação interpretativa da similaridade encontrada pelo modelo.

O objetivo é transformar o valor matemático de similaridade em uma informação mais compreensível para o usuário final.

Critérios:

- Acima de 80%: Muito relacionada;
- Entre 60% e 79%: Relacionada;
- Entre 35% e 59%: Possível relação.

Essa camada facilita a interpretação dos resultados apresentados pelo SCRIPTO.

In [ ]:
def classificar_relevancia(score):

    if score >= 80:
        return "Muito relacionada"

    elif score >= 60:
        return "Relacionada"

    else:
        return "Possível relação"


def exibir_recomendacoes(resultados):

    print("🔎 Conteúdos relacionados encontrados\n")

    print("=" * 50)

    for posicao, item in enumerate(resultados, start=1):

        if posicao == 1:
            medalha = "🥇"
        elif posicao == 2:
            medalha = "🥈"
        elif posicao == 3:
            medalha = "🥉"
        else:
            medalha = "📌"

        relevancia = classificar_relevancia(item["similaridade"])

        print(f"{medalha} {posicao}º Resultado")
        print(f"📂 Projeto: {item['projeto']}")
        print(f"📄 Documento: {item['documento']}")
        print(f"🏷 Categoria: {item['categoria']}")
        print(f"⭐ Similaridade: {item['similaridade']}%")
        print(f"📌 Relevância: {relevancia}")
        print("-" * 50)

# Refinamento da Apresentação dos Resultados - SCRIPTO

Nesta etapa será aprimorada a apresentação das recomendações geradas pelo modelo de IA.

O objetivo é transformar os valores de similaridade em níveis de relevância mais intuitivos para o usuário final.

Também será apresentado o documento consultado antes da lista de recomendações, criando uma experiência mais próxima da utilização real da plataforma SCRIPTO.

In [ ]:
def classificar_relevancia(score):

    if score >= 90:
        return "🟢 Altamente relacionada"

    elif score >= 70:
        return "🟡 Relacionada"

    elif score >= 50:
        return "🔵 Moderadamente relacionada"

    else:
        return "⚪ Baixa relação"

# Expansão da Base de Conhecimento - Todos os Documentos Técnicos

Inicialmente o modelo foi desenvolvido utilizando apenas arquivos README.md, devido à concentração de informações descritivas nesses documentos.

Nesta etapa, a base de conhecimento será expandida para incluir todos os documentos técnicos disponíveis no dataset tratado.

Serão considerados documentos como:

- README.md;
- LICENSE;
- CONTRIBUTING.md;
- CHANGELOG.md;
- SECURITY.md;
- arquivos de configuração de dependências e linguagens.

Objetivo:

Aumentar a capacidade do SCRIPTO de encontrar, classificar e recomendar conteúdos técnicos relacionados, permitindo uma visão mais ampla do conhecimento disponível nos repositórios analisados.

# Análise da Base Completa de Documentos Técnicos

Nesta etapa será analisada a distribuição dos documentos após a expansão da base de conhecimento.

A validação permite verificar quais tipos de conteúdos técnicos serão utilizados pelo modelo de IA.

A base contempla diferentes categorias, como documentação, licenças, dependências e arquivos de configuração.

In [ ]:
dataset["categoria_documento"].value_counts()

# Geração de Embeddings da Base Completa - SCRIPTO

Nesta etapa será aplicado o modelo de linguagem Sentence Transformer em todos os documentos técnicos tratados.

Cada documento será convertido em um vetor numérico (embedding), permitindo que o sistema compare conteúdos por similaridade semântica.

Processo:

1. Leitura dos 2.879 documentos técnicos;
2. Conversão do conteúdo textual em embeddings;
3. Armazenamento dos vetores gerados;
4. Salvamento dos metadados para consulta futura.

Resultado esperado:

- 2.879 embeddings gerados;
- Vetores com 384 dimensões;
- Base preparada para busca semântica e recomendação automática.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Carregar modelo
modelo = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Modelo carregado!")

In [ ]:
# Gerar embeddings de todos os documentos

textos = dataset["content"].tolist()

embeddings_documentos = modelo.encode(
    textos,
    batch_size=32,
    show_progress_bar=True
)

print("Embeddings gerados!")
print("Quantidade de documentos:", len(embeddings_documentos))
print("Dimensão dos vetores:", embeddings_documentos.shape[1])

# Armazenamento dos Embeddings e Metadados - SCRIPTO

Após a geração dos embeddings, os vetores serão armazenados para evitar o processamento novamente a cada execução.

Também serão salvos os metadados dos documentos, permitindo relacionar cada vetor ao seu repositório, arquivo e categoria.

Arquivos gerados:

- embeddings_documentos_tecnicos.npy
  - Contém os vetores numéricos gerados pelo modelo de IA.

- documentos_tecnicos.csv
  - Contém as informações dos documentos para consulta e identificação dos resultados.

In [ ]:
import os

pasta_embeddings = "/content/drive/MyDrive/Hackathon_ONE/SCRIPTO_AI/embeddings"
pasta_metadados = "/content/drive/MyDrive/Hackathon_ONE/SCRIPTO_AI/metadados"

os.makedirs(pasta_embeddings, exist_ok=True)
os.makedirs(pasta_metadados, exist_ok=True)

print("Estrutura SCRIPTO_AI criada!")

In [ ]:
caminho_embeddings = (
    "/content/drive/MyDrive/Hackathon_ONE/"
    "SCRIPTO_AI/embeddings/"
    "embeddings_documentos_tecnicos.npy"
)

np.save(
    caminho_embeddings,
    embeddings_documentos
)

print("Embeddings salvos com sucesso!")
print(caminho_embeddings)

In [ ]:
metadados_documentos = dataset[
    [
        "repo_full_name",
        "file_name",
        "categoria_documento",
        "num_caracteres"
    ]
].copy()


caminho_metadados = (
    "/content/drive/MyDrive/Hackathon_ONE/"
    "SCRIPTO_AI/metadados/"
    "documentos_tecnicos.csv"
)


metadados_documentos.to_csv(
    caminho_metadados,
    index=False
)

print("Metadados salvos com sucesso!")
print(metadados_documentos.shape)

In [ ]:
import os

print(os.path.exists(caminho_embeddings))
print(os.path.exists(caminho_metadados))

# Carregamento da Base Completa de IA - SCRIPTO

Nesta etapa será carregada a nova base de conhecimento contendo todos os documentos técnicos processados.

A versão anterior utilizava apenas arquivos README.md.

Nesta versão, o SCRIPTO utiliza todos os documentos classificados:

- README.md;
- LICENSE;
- CONTRIBUTING.md;
- CHANGELOG.md;
- SECURITY.md;
- arquivos de dependências;
- arquivos de configuração.

Essa expansão aumenta a capacidade de busca e recomendação semântica do sistema.

In [ ]:
import numpy as np
import pandas as pd

# Caminhos da nova base

caminho_embeddings = (
    "/content/drive/MyDrive/Hackathon_ONE/"
    "SCRIPTO_AI/embeddings/"
    "embeddings_documentos_tecnicos.npy"
)

caminho_metadados = (
    "/content/drive/MyDrive/Hackathon_ONE/"
    "SCRIPTO_AI/metadados/"
    "documentos_tecnicos.csv"
)


# Carregar embeddings
embeddings_base = np.load(caminho_embeddings)


# Carregar metadados
metadados = pd.read_csv(caminho_metadados)


print("Base completa carregada!")
print("Embeddings:", embeddings_base.shape)
print("Metadados:", metadados.shape)

# Busca Semântica na Base Completa - SCRIPTO

Nesta etapa será atualizada a função de busca inteligente utilizando todos os documentos técnicos disponíveis.

O usuário poderá realizar consultas em linguagem natural e o sistema irá comparar o significado da pesquisa com os embeddings de todos os documentos.

A busca considera diferentes tipos de arquivos técnicos, permitindo encontrar conteúdos relacionados mesmo quando os termos utilizados são diferentes.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity


def buscar_scripto(pergunta, top_n=5):

    # Transformar pergunta em embedding
    embedding_pergunta = modelo.encode([pergunta])


    # Comparar com todos os documentos
    similaridades = cosine_similarity(
        embedding_pergunta,
        embeddings_base
    )[0]


    # Selecionar maiores similaridades
    indices = np.argsort(similaridades)[::-1][:top_n]


    resultados = []

    for indice in indices:

        resultados.append({
            "projeto": metadados.iloc[indice]["repo_full_name"],
            "documento": metadados.iloc[indice]["file_name"],
            "categoria": metadados.iloc[indice]["categoria_documento"],
            "similaridade": round(
                float(similaridades[indice]) * 100,
                2
            )
        })


    return resultados

In [ ]:
buscar_scripto(
    "Como instalar dependências de um projeto Python"
)

In [ ]:
buscar_scripto(
    "Como contribuir com um projeto open source"
)

In [ ]:
metadados["categoria_documento"].value_counts()

# Ranking Inteligente por Categoria - SCRIPTO

Nesta etapa será adicionada uma camada de inteligência ao ranking dos resultados.

Além da similaridade semântica calculada pelo modelo de IA, serão considerados os tipos de documentos encontrados.

O objetivo é aumentar a relevância dos resultados de acordo com a intenção da busca.

Exemplo:

Uma busca relacionada a contribuição em projetos open source deve priorizar:

- CONTRIBUTING.md;
- CODE_OF_CONDUCT.md;
- documentos de Governança.

Essa abordagem combina inteligência artificial com regras de negócio para melhorar a experiência do usuário.

In [ ]:
def aplicar_peso_categoria(categoria, pergunta):

    pergunta = pergunta.lower()

    peso = 1.0


    # Busca relacionada a contribuição
    if any(
        palavra in pergunta
        for palavra in [
            "contribuir",
            "contribuição",
            "open source",
            "colaborar"
        ]
    ):

        if categoria == "Contribuicao":
            peso = 1.25

        elif categoria == "Governanca":
            peso = 1.15

        elif categoria == "Licenca":
            peso = 0.90


    # Busca relacionada a dependências
    elif any(
        palavra in pergunta
        for palavra in [
            "dependência",
            "instalar",
            "pacote",
            "biblioteca"
        ]
    ):

        if categoria == "Dependencias":
            peso = 1.25


    return peso

In [ ]:
def buscar_scripto_inteligente(pergunta, top_n=5):

    embedding_pergunta = modelo.encode([pergunta])

    similaridades = cosine_similarity(
        embedding_pergunta,
        embeddings_base
    )[0]


    resultados = []


    for indice, similaridade in enumerate(similaridades):

        categoria = metadados.iloc[indice]["categoria_documento"]

        peso = aplicar_peso_categoria(
            categoria,
            pergunta
        )


        score_final = similaridade * peso * 100


        resultados.append({

            "projeto": metadados.iloc[indice]["repo_full_name"],
            "documento": metadados.iloc[indice]["file_name"],
            "categoria": categoria,
            "similaridade": round(float(score_final),2)

        })


    resultados = sorted(
        resultados,
        key=lambda x: x["similaridade"],
        reverse=True
    )


    return resultados[:top_n]

In [ ]:
buscar_scripto_inteligente(
    "Como contribuir com um projeto open source"
)

# Salvamento da Base Final de IA - SCRIPTO

Nesta etapa são armazenados os artefatos finais gerados pelo modelo de IA.

Esses arquivos serão utilizados posteriormente pelo Backend para integração com a aplicação.

Arquivos gerados:

- embeddings_documentos_tecnicos.npy:
  Contém os vetores semânticos dos documentos.

- documentos_tecnicos.csv:
  Contém os metadados utilizados para identificar os resultados.

Esses arquivos representam a base de conhecimento do SCRIPTO.

In [ ]:
import os

pasta_final = "/content/drive/MyDrive/Hackathon_ONE/SCRIPTO_AI/modelo_final"

os.makedirs(
    pasta_final,
    exist_ok=True
)

print("Pasta final criada!")

In [ ]:
np.save(
    "/content/drive/MyDrive/Hackathon_ONE/SCRIPTO_AI/modelo_final/embeddings_scripto.npy",
    embeddings_base
)

print("Embeddings finais salvos!")

In [ ]:
metadados.to_csv(
    "/content/drive/MyDrive/Hackathon_ONE/SCRIPTO_AI/modelo_final/metadados_scripto.csv",
    index=False
)

print("Metadados finais salvos!")

# Validação Final do Modelo

Nesta etapa são realizados testes para verificar se o modelo consegue recuperar documentos relacionados a diferentes intenções de busca.

Os testes simulam possíveis consultas realizadas pelos usuários do SCRIPTO.

In [ ]:
buscar_scripto_inteligente(
    "Como instalar dependências de um projeto Python"
)

In [ ]:
buscar_scripto_inteligente(
    "Como configurar uma aplicação Java"
)

# Resumo Técnico - Data/IA SCRIPTO

O módulo de Inteligência Artificial do SCRIPTO foi desenvolvido utilizando técnicas de Processamento de Linguagem Natural (PLN) e embeddings semânticos.

Processamento realizado:

- Limpeza e preparação de 2.879 documentos técnicos;
- Classificação automática por categoria;
- Geração de embeddings utilizando Sentence Transformer;
- Criação de mecanismo de busca semântica;
- Desenvolvimento de ranking inteligente considerando similaridade e categoria documental.

Resultado:

O modelo permite encontrar conteúdos técnicos relacionados considerando o significado dos textos, indo além de buscas tradicionais por palavras-chave.

Artefatos disponibilizados para integração:

- embeddings_scripto.npy
- metadados_scripto.csv

# Evolução do Modelo de IA

A partir desta etapa, o modelo de IA passa a atender ao contrato definido para integração com o Backend (Spring Boot).

## Objetivo

Receber um conteúdo técnico enviado pelo usuário e retornar uma resposta estruturada contendo:

- Categoria principal do documento;
- Probabilidade (confiança) da classificação;
- Até 5 tags relevantes;
- Resumo automático (máximo de 20 palavras);
- Nível estimado do conteúdo:
  - Iniciante
  - Intermediário
  - Avançado

A saída será entregue em formato JSON para consumo da API do projeto SCRIPTO.

In [ ]:
import numpy as np
import pandas as pd

caminho_embeddings = "/content/drive/MyDrive/Hackathon_ONE/SCRIPTO_AI/modelo_final/embeddings_scripto.npy"
caminho_metadados = "/content/drive/MyDrive/Hackathon_ONE/SCRIPTO_AI/modelo_final/metadados_scripto.csv"

embeddings_base = np.load(caminho_embeddings)
metadados = pd.read_csv(caminho_metadados)

print("Embeddings:", embeddings_base.shape)
print("Metadados:", metadados.shape)
print(metadados["categoria_documento"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X = embeddings_base
y = metadados["categoria_documento"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

classificador = LogisticRegression(max_iter=1000)
classificador.fit(X_train, y_train)

y_pred = classificador.predict(X_test)

print(classification_report(y_test, y_pred))

In [ ]:
# ============================================================
# PREVISÃO DA CATEGORIA
# ============================================================

def prever_categoria(texto):
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError("O conteúdo enviado deve ser um texto não vazio.")

    embedding_texto = modelo.encode([texto])

    categoria_prevista = classificador.predict(
        embedding_texto
    )[0]

    probabilidades = classificador.predict_proba(
        embedding_texto
    )[0]

    probabilidade_maxima = float(
        probabilidades.max()
    )

    return {
        "categoria": categoria_prevista,
        "probabilidade": round(probabilidade_maxima, 2)
    }

In [ ]:
# ============================================================
# EXTRAÇÃO DE TAGS
# ============================================================

import re
from collections import Counter

PALAVRAS_IGNORADAS = {
    "a", "o", "as", "os", "um", "uma", "uns", "umas",
    "de", "da", "do", "das", "dos", "em", "no", "na",
    "nos", "nas", "para", "por", "com", "sem", "sobre",
    "e", "ou", "que", "se", "como", "mais", "menos",
    "este", "esta", "esse", "essa", "isso", "são", "ser",
    "foi", "tem", "ter", "usando", "utiliza", "utilizado",
    "utilizando", "projeto", "arquivo", "documento", "conteúdo",
    "criar", "sistema", "realiza", "todo", "todos"
}


def extrair_tags(texto, limite=5):
    if not isinstance(texto, str) or not texto.strip():
        return []

    palavras = re.findall(
        r"[a-zA-ZÀ-ÿ][a-zA-ZÀ-ÿ0-9+#.\-]{2,}",
        texto.lower()
    )

    palavras_validas = []

    for palavra in palavras:
        palavra_limpa = palavra.strip(".,;:!?()[]{}")

        if (
            palavra_limpa
            and palavra_limpa not in PALAVRAS_IGNORADAS
            and len(palavra_limpa) > 2
        ):
            palavras_validas.append(palavra_limpa)

    frequencias = Counter(palavras_validas)

    return [
        palavra
        for palavra, _ in frequencias.most_common(limite)
    ]

In [ ]:
print(extrair_tags(texto_teste))

In [ ]:
# ============================================================
# RESUMO AUTOMÁTICO
# ============================================================

import re

def gerar_resumo(texto, limite_palavras=20):
    """
    Gera um resumo simples utilizando as primeiras palavras do texto.
    """

    if not isinstance(texto, str) or not texto.strip():
        return ""

    # Remove espaços duplicados
    texto = re.sub(r"\s+", " ", texto).strip()

    palavras = texto.split()

    resumo = " ".join(palavras[:limite_palavras])

    if len(palavras) > limite_palavras:
        resumo += "..."

    return resumo

In [ ]:
print(gerar_resumo(texto_teste))

In [ ]:
# ============================================================
# ESTIMATIVA DO NÍVEL DO CONTEÚDO
# ============================================================

TERMOS_AVANCADOS = {
    "arquitetura", "microsserviços", "microservices",
    "escalabilidade", "concorrência", "orquestração",
    "kubernetes", "docker", "pipeline", "ci/cd",
    "machine learning", "deep learning", "embedding",
    "transformer", "distribuído", "oauth", "jwt"
}

TERMOS_INTERMEDIARIOS = {
    "api", "rest", "framework", "spring", "django",
    "flask", "banco", "database", "sql", "maven",
    "gradle", "dependência", "dependências", "teste",
    "deploy", "backend", "frontend", "configuração"
}


def estimar_nivel(texto):
    if not isinstance(texto, str) or not texto.strip():
        return "Iniciante"

    texto_normalizado = texto.lower()
    quantidade_palavras = len(texto_normalizado.split())

    pontos_avancado = sum(
        termo in texto_normalizado
        for termo in TERMOS_AVANCADOS
    )

    pontos_intermediario = sum(
        termo in texto_normalizado
        for termo in TERMOS_INTERMEDIARIOS
    )

    if pontos_avancado >= 2 or quantidade_palavras > 250:
        return "Avançado"

    if (
        pontos_intermediario >= 2
        or pontos_avancado >= 1
        or quantidade_palavras > 80
    ):
        return "Intermediário"

    return "Iniciante"

In [ ]:
print(estimar_nivel(texto_teste))

In [ ]:
# ============================================================
# FUNÇÃO PRINCIPAL DO SCRIPTO
# ============================================================

def analisar_documento(texto):
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError("O conteúdo enviado deve ser um texto não vazio.")

    previsao = prever_categoria(texto)

    resultado = {
        "categoria": previsao["categoria"],
        "probabilidade": previsao["probabilidade"],
        "tags": extrair_tags(texto, limite=5),
        "resumo": gerar_resumo(texto, limite_palavras=20),
        "nivel": estimar_nivel(texto).upper()
    }

    return resultado

In [ ]:
resultado = analisar_documento(texto_teste)

print(resultado)

In [ ]:
# ============================================================
# SAÍDA JSON PARA O BACKEND
# ============================================================

import json

def analisar_documento_json(texto):
    resultado = analisar_documento(texto)

    return json.dumps(
        resultado,
        ensure_ascii=False,
        indent=4
    )

In [ ]:
resultado_json = analisar_documento_json(texto_teste)

print(resultado_json)

In [ ]:
# ============================================================
# GERAÇÃO DO MOCK PARA O BACKEND
# ============================================================

import os

# Caminho onde o mock será salvo
CAMINHO_MOCK = "/content/drive/MyDrive/Hackathon_ONE/mock_classificador.json"

# Gera o JSON utilizando o modelo
resultado_json = analisar_documento_json(texto_teste)

# Salva o arquivo
with open(CAMINHO_MOCK, "w", encoding="utf-8") as arquivo:
    arquivo.write(resultado_json)

print("✅ Mock criado com sucesso!")
print(f"📁 Local: {CAMINHO_MOCK}")

# SCRIPTO - Módulo de Inteligência Artificial

## Objetivo

O módulo de Inteligência Artificial do projeto SCRIPTO é responsável por analisar conteúdos técnicos enviados pelo usuário e retornar informações estruturadas para consumo pelo Backend (Spring Boot).

A saída do modelo é padronizada em formato JSON, permitindo sua integração com os demais componentes do sistema.

---

# Fluxo da IA

Entrada (Texto Técnico)
        ↓
Pré-processamento
        ↓
Geração de Embeddings
        ↓
Classificação do Documento
        ↓
Extração de Tags
        ↓
Geração de Resumo
        ↓
Estimativa de Nível
        ↓
Saída JSON

---

# Tecnologias Utilizadas

- Python
- Pandas
- Sentence Transformers
- Scikit-Learn
- JSON
- Google Colab

---

# Funcionalidades Implementadas

O módulo realiza automaticamente:

- Classificação da categoria do documento;
- Cálculo da probabilidade da classificação;
- Extração de até 5 palavras-chave (tags);
- Geração de resumo automático;
- Estimativa do nível do conteúdo (INICIANTE, INTERMEDIÁRIO ou AVANÇADO);
- Geração de resposta estruturada em JSON.

---

# Contrato de Saída

Exemplo de resposta:

```json
{
    "categoria": "Documentacao",
    "probabilidade": 0.92,
    "tags": [
        "spring",
        "boot",
        "api",
        "rest",
        "maven"
    ],
    "resumo": "Este projeto utiliza Spring Boot para criar uma API REST...",
    "nivel": "INTERMEDIÁRIO"
}
```

---

# Integração com o Backend

O Backend (Spring Boot) deverá enviar um texto técnico para o módulo de IA.

A IA processará o conteúdo e retornará um JSON contendo todas as informações necessárias para a aplicação.

---

# Estrutura da IA

O notebook `03_Modelo_IA_SCRIPTO.ipynb` contém:

- treinamento do classificador;
- geração de embeddings;
- classificação dos documentos;
- extração de tags;
- geração de resumo;
- estimativa de nível;
- geração do JSON;
- criação do mock (`mock_classificador.json`).

---

# Mock para Desenvolvimento

Foi disponibilizado o arquivo:

```
mock_classificador.json
```

Esse arquivo permite que a equipe de Backend desenvolva e teste as integrações mesmo sem executar o modelo de IA.

---

# Autora

Juliana Ferreira dos Santos Magalhães

Hackathon ONE – Oracle + Alura

# ============================================================
# EXEMPLOS DE UTILIZAÇÃO DA IA
# ============================================================

Nesta seção são apresentados exemplos práticos de utilização do módulo de Inteligência Artificial desenvolvido para o projeto SCRIPTO.

Cada exemplo envia um conteúdo técnico para a IA e recebe uma resposta estruturada em formato JSON, conforme o contrato definido para integração com o Backend (Spring Boot).

In [ ]:
# ============================================================
# EXEMPLO 1 - CONTEÚDO SOBRE JAVA E SPRING BOOT
# ============================================================

texto_exemplo_java = """
Projeto desenvolvido com Java e Spring Boot para criação de uma API REST.
O Maven é utilizado para gerenciamento de dependências e o sistema possui
autenticação JWT para proteger os endpoints.
"""

resultado_exemplo_java = analisar_documento_json(texto_exemplo_java)

print(resultado_exemplo_java)

# ============================================================
# EXEMPLO 2 - CONTEÚDO SOBRE PYTHON E CIÊNCIA DE DADOS
# ============================================================

texto_exemplo_python = """
Aplicação desenvolvida em Python utilizando Pandas e NumPy para análise exploratória de dados.
Os resultados são apresentados em gráficos e tabelas para apoiar a tomada de decisão.
"""

resultado_exemplo_python = analisar_documento_json(texto_exemplo_python)

print(resultado_exemplo_python)

In [ ]:
# ============================================================
# EXEMPLO 2 - CONTEÚDO SOBRE PYTHON E CIÊNCIA DE DADOS
# ============================================================

texto_exemplo_python = """
Aplicação desenvolvida em Python utilizando Pandas e NumPy para análise exploratória de dados.
Os resultados são apresentados em gráficos e tabelas para apoiar a tomada de decisão.
"""

resultado_exemplo_python = analisar_documento_json(texto_exemplo_python)

print(resultado_exemplo_python)

## Conclusão

Os exemplos apresentados demonstram o funcionamento do módulo de Inteligência Artificial desenvolvido para o projeto SCRIPTO.

A função `analisar_documento_json()` representa o ponto de integração entre a camada de IA e o Backend (Spring Boot), retornando todas as informações necessárias em formato JSON conforme o contrato definido pela equipe.

O arquivo `mock_classificador.json` foi disponibilizado para permitir que a equipe de Backend realize testes de integração independentemente da execução do modelo.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# @title
!find /content/drive/MyDrive -name "*Modelo*IA*.ipynb"

In [ ]:
!git checkout develop-data

In [ ]:
!git branch

In [ ]:
!ls

In [ ]:
import os

os.makedirs("data/ia", exist_ok=True)

print("Estrutura criada!")

In [ ]:
!ls data/ia

In [ ]:
!ls -la

In [ ]:
!ls -la data

In [ ]:
!cp "/content/drive/MyDrive/Hackathon_ONE/03_Modelo_IA_SCRIPTO.ipynb" "data/ia/"

In [ ]:
!ls -la data/ia

In [ ]:
!git status

In [ ]:
!git add data/

In [ ]:
!git push origin develop-data

In [ ]:
!git log -1

In [ ]:
!git config --global user.name "Juliana Ferreira"

In [ ]:
# @title
from getpass import getpass

token = getpass("Digite seu token do GitHub:")